# 從零開始寫一個 GPT：Self-Attention 與 Transformer

<a href="https://colab.research.google.com/github/tjwei/DataScienceSoftware2026/blob/main/05-pytorch/05-Attention-and-GPT.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

建議環境：Google Colab，`執行階段 → 變更執行階段類型 → T4 GPU`

In [ ]:
# 下載本章練習的參考解答 (q_*.py)
import os, urllib.request
BASE_URL = 'https://raw.githubusercontent.com/tjwei/DataScienceSoftware2026/main/05-pytorch/'
for fn in ['q_attention.py', 'q_topk.py']:
    if not os.path.exists(fn):
        urllib.request.urlretrieve(BASE_URL + fn, fn)

In [ ]:
import math, time
import torch
import torch.nn as nn
import torch.nn.functional as F
device = "cuda" if torch.cuda.is_available() else "cpu"
torch.manual_seed(42)

## 資料：Tiny Shakespeare

我們要訓練一個**字元層級 (character-level)** 的語言模型：給定前面的字，預測下一個字。
這就是 ChatGPT 等大型語言模型 (LLM) 的基本原理，只是規模小很多。

In [ ]:
from urllib.request import urlopen
url = "https://raw.githubusercontent.com/karpathy/char-rnn/master/data/tinyshakespeare/input.txt"
text = urlopen(url).read().decode("utf8")
print(len(text))
print(text[:300])

### Tokenizer

最簡單的 tokenizer：每個字元一個編號。（真正的 LLM 用 BPE 等方法，下一個單元會看到）

In [ ]:
chars = sorted(set(text))
vocab_size = len(chars)
stoi = {c: i for i, c in enumerate(chars)}
itos = {i: c for i, c in enumerate(chars)}
encode = lambda s: [stoi[c] for c in s]
decode = lambda l: "".join(itos[i] for i in l)
print(vocab_size, repr("".join(chars)))
print(encode("Hello"), decode(encode("Hello")))

In [ ]:
data = torch.tensor(encode(text), dtype=torch.long)
n = int(0.9 * len(data))
train_data, val_data = data[:n], data[n:]

def get_batch(split, batch_size, block_size):
    d = train_data if split == "train" else val_data
    ix = torch.randint(len(d) - block_size - 1, (batch_size,))
    x = torch.stack([d[i:i + block_size] for i in ix])
    y = torch.stack([d[i + 1:i + block_size + 1] for i in ix])   # 目標是往後移一格
    return x.to(device), y.to(device)

x, y = get_batch("train", 2, 16)
for t in range(5):
    print(repr(decode(x[0, :t + 1].tolist())), "->", repr(itos[y[0, t].item()]))

## Baseline：Bigram model

只看前一個字來預測下一個字。整個模型就是一個 `vocab_size x vocab_size` 的表格 (`nn.Embedding`)。

In [ ]:
class Bigram(nn.Module):
    def __init__(self):
        super().__init__()
        self.table = nn.Embedding(vocab_size, vocab_size)
    def forward(self, idx, targets=None):
        logits = self.table(idx)                     # (B, T, vocab)
        loss = None if targets is None else F.cross_entropy(logits.view(-1, vocab_size), targets.view(-1))
        return logits, loss

@torch.no_grad()
def generate(model, idx, max_new_tokens, temperature=1.0, block_size=None):
    model.eval()
    for _ in range(max_new_tokens):
        idx_cond = idx if block_size is None else idx[:, -block_size:]
        logits, _ = model(idx_cond)
        logits = logits[:, -1, :].float() / temperature   # 只要最後一個位置的預測
        probs = F.softmax(logits, dim=-1)
        next_id = torch.multinomial(probs, num_samples=1)  # 依機率抽樣
        idx = torch.cat([idx, next_id], dim=1)
    return idx

bigram = Bigram().to(device)
opt = torch.optim.AdamW(bigram.parameters(), lr=1e-2)
for step in range(3000):
    xb, yb = get_batch("train", 64, 32)
    _, loss = bigram(xb, yb)
    opt.zero_grad(); loss.backward(); opt.step()
print("loss", loss.item())
start = torch.tensor([encode("\n")], device=device)
print(decode(generate(bigram, start, 300)[0].tolist()))

## Self-Attention

Bigram 只看前一個字。我們希望每個位置可以「參考」前面**所有**位置的資訊。

最簡單的想法：取前面所有位置的平均。用矩陣乘法 + 下三角遮罩 (causal mask) 可以一次算完：

In [ ]:
T = 5
mask = torch.tril(torch.ones(T, T))
w = torch.zeros(T, T).masked_fill(mask == 0, float("-inf"))
w = F.softmax(w, dim=-1)
w

平均是把每個位置看成一樣重要。**Attention** 則讓權重由資料決定：

* 每個位置產生 query $q$、key $k$、value $v$ 三個向量
* 權重 = $\mathrm{softmax}\left(\frac{QK^T}{\sqrt{d}}\right)$（加上 causal mask，不能偷看未來）
* 輸出 = 權重 $\times V$

$$\mathrm{Attention}(Q,K,V) = \mathrm{softmax}\left(\frac{QK^T}{\sqrt{d}} + M\right)V$$

## Q
用基本的矩陣運算寫出 causal attention，並和 PyTorch 內建的
`F.scaled_dot_product_attention(q, k, v, is_causal=True)` 比較結果。

* `q, k, v` 的形狀都是 `(B, n_head, T, head_dim)`
* 內建的 SDPA 會自動挑選最快的實作（FlashAttention、memory-efficient attention 等），請盡量用它

In [ ]:
# %load q_attention.py
%run -i q_attention.py

## Transformer Block

現代 LLM (Llama, Qwen, ...) 的 block 大致是：

```
x = x + Attention(RMSNorm(x))
x = x + MLP(RMSNorm(x))
```

* **Multi-head**：把向量切成好幾份，各自做 attention
* **Residual connection** + **Pre-normalization**：讓深層網路好訓練
* 真正的 LLM 還會用 RoPE 位置編碼、SwiGLU、GQA 等改進，這裡先用最簡單的版本

In [ ]:
class CausalSelfAttention(nn.Module):
    def __init__(self, n_embd, n_head, dropout):
        super().__init__()
        self.n_head = n_head
        self.qkv = nn.Linear(n_embd, 3 * n_embd, bias=False)
        self.proj = nn.Linear(n_embd, n_embd, bias=False)
        self.dropout = dropout
    def forward(self, x):
        B, T, C = x.shape
        q, k, v = self.qkv(x).split(C, dim=2)
        # (B, T, C) -> (B, n_head, T, head_dim)
        q, k, v = [t.view(B, T, self.n_head, C // self.n_head).transpose(1, 2) for t in (q, k, v)]
        y = F.scaled_dot_product_attention(q, k, v, is_causal=True,
                                           dropout_p=self.dropout if self.training else 0.0)
        y = y.transpose(1, 2).contiguous().view(B, T, C)
        return self.proj(y)

class MLP(nn.Module):
    def __init__(self, n_embd, dropout):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(n_embd, 4 * n_embd, bias=False), nn.GELU(),
            nn.Linear(4 * n_embd, n_embd, bias=False), nn.Dropout(dropout))
    def forward(self, x):
        return self.net(x)

class Block(nn.Module):
    def __init__(self, n_embd, n_head, dropout):
        super().__init__()
        self.norm1 = nn.RMSNorm(n_embd)
        self.attn = CausalSelfAttention(n_embd, n_head, dropout)
        self.norm2 = nn.RMSNorm(n_embd)
        self.mlp = MLP(n_embd, dropout)
    def forward(self, x):
        x = x + self.attn(self.norm1(x))
        x = x + self.mlp(self.norm2(x))
        return x

class GPT(nn.Module):
    def __init__(self, vocab_size, block_size, n_layer=6, n_head=6, n_embd=384, dropout=0.1):
        super().__init__()
        self.block_size = block_size
        self.tok_emb = nn.Embedding(vocab_size, n_embd)
        self.pos_emb = nn.Embedding(block_size, n_embd)   # 可學習的位置編碼
        self.blocks = nn.Sequential(*[Block(n_embd, n_head, dropout) for _ in range(n_layer)])
        self.norm = nn.RMSNorm(n_embd)
        self.lm_head = nn.Linear(n_embd, vocab_size, bias=False)
        self.lm_head.weight = self.tok_emb.weight          # weight tying
        self.apply(self._init_weights)
    def _init_weights(self, m):
        if isinstance(m, (nn.Linear, nn.Embedding)):
            nn.init.normal_(m.weight, std=0.02)
    def forward(self, idx, targets=None):
        B, T = idx.shape
        pos = torch.arange(T, device=idx.device)
        x = self.tok_emb(idx) + self.pos_emb(pos)
        x = self.blocks(x)
        logits = self.lm_head(self.norm(x))
        loss = None if targets is None else F.cross_entropy(logits.view(-1, logits.size(-1)).float(), targets.view(-1))
        return logits, loss

In [ ]:
block_size = 256
model = GPT(vocab_size, block_size, n_layer=6, n_head=6, n_embd=384, dropout=0.2).to(device)
print(f"參數數量: {sum(p.numel() for p in model.parameters())/1e6:.2f}M")

## 訓練

用 fp16 混合精度。在 T4 上大約需要 3 分鐘。

注意觀察 `train` 和 `val` loss：如果 train 一直下降、val 卻開始上升，就是 **overfitting**（模型開始背誦訓練資料）。試試看把 `max_steps` 加大到 3000、或把 `dropout` 改成 0.0，會發生什麼事？

In [ ]:
@torch.no_grad()
def estimate_loss(model, iters=20, batch_size=64):
    model.eval()
    out = {}
    for split in ["train", "val"]:
        losses = []
        for _ in range(iters):
            xb, yb = get_batch(split, batch_size, block_size)
            with torch.autocast(device, dtype=torch.float16):
                _, loss = model(xb, yb)
            losses.append(loss.item())
        out[split] = sum(losses) / len(losses)
    model.train()
    return out

max_steps, batch_size, lr = 1500, 64, 1e-3
optimizer = torch.optim.AdamW(model.parameters(), lr=lr, betas=(0.9, 0.99), weight_decay=0.1)
scheduler = torch.optim.lr_scheduler.OneCycleLR(optimizer, max_lr=lr, total_steps=max_steps, pct_start=0.05)
scaler = torch.amp.GradScaler(device)
t0 = time.perf_counter()
model.train()
for step in range(max_steps + 1):
    if step % 250 == 0:
        losses = estimate_loss(model)
        print(f"step {step:5d}  train {losses['train']:.3f}  val {losses['val']:.3f}  {time.perf_counter()-t0:.0f}s")
    if step == max_steps:
        break
    xb, yb = get_batch("train", batch_size, block_size)
    with torch.autocast(device, dtype=torch.float16):
        _, loss = model(xb, yb)
    optimizer.zero_grad(set_to_none=True)
    scaler.scale(loss).backward()
    scaler.unscale_(optimizer)
    torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)   # gradient clipping
    scaler.step(optimizer)
    scaler.update()
    scheduler.step()

In [ ]:
start = torch.tensor([encode("ROMEO:\n")], device=device)
with torch.autocast(device, dtype=torch.float16):
    print(decode(generate(model, start, 500, temperature=0.8, block_size=block_size)[0].tolist()))

## 溫度 (temperature)

`temperature` 越低，模型越保守（重複、無聊）；越高越有創意（但也越容易胡言亂語）。

In [ ]:
for t in [0.3, 1.0, 1.5]:
    print(f"===== temperature={t}")
    with torch.autocast(device, dtype=torch.float16):
        print(decode(generate(model, start, 200, temperature=t, block_size=block_size)[0].tolist()))

## Q：Top-k sampling
修改 `generate`，只從機率最高的 `k` 個字中抽樣（其他的 logits 設成 `-inf`）。

提示：`torch.topk(logits, k)`

In [ ]:
# %load q_topk.py
%run -i q_topk.py

## 思考

* 現在每生成一個字，就要把整段文字重新算一遍。實際的 LLM 會用 **KV cache** 把之前算過的 key, value 存起來（下一單元的 🤗 transformers 會幫我們處理）
* 我們的模型 ~10M 參數、字元層級；Qwen3-0.6B 有 6 億參數、詞彙量 15 萬，用了 36 兆 tokens 訓練
* 試試看用別的文字（例如中文小說、自己的程式碼）來訓練